# 07 — Linear Regression Fresh Assessment (Section 15A)

**Exercise A: Independent implementation (fresh notebook)**

Implements linear regression from scratch without looking at previous notebooks:
- Generate data
- Initialize parameters
- Calculate predictions
- Calculate loss
- Backpropagate
- Update parameters
- Evaluate the final model

**Compliance:** no sklearn LinearRegression, no `torch.optim.*`, no `torch.nn.*` modules/losses/functional helpers, no `torch.utils.data` abstractions. Raw tensors + autograd + manual updates only.


In [1]:
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

torch.manual_seed(123)
np.random.seed(123)
os.makedirs('../reports', exist_ok=True)

# 1. Generate data
n = 100
true_w = 2.0
true_b = -0.5
x = torch.linspace(-3, 3, n).unsqueeze(1)
y = true_w * x + true_b + torch.randn(n, 1) * 0.4

idx = torch.randperm(n)
n_train = 80
x_train, y_train = x[idx[:n_train]], y[idx[:n_train]]
x_val, y_val = x[idx[n_train:]], y[idx[n_train:]]

# 2. Initialize parameters
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# 3-6. prediction, loss, backprop, update
def predict(x_in, w_in, b_in):
  return x_in * w_in + b_in

def mse(y_hat, y_true):
  return ((y_hat - y_true) ** 2).mean()

lr = 0.05
epochs = 300
train_losses = []
val_losses = []
for _ in range(epochs):
  y_hat = predict(x_train, w, b)
  loss = mse(y_hat, y_train)
  loss.backward()
  with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()
  train_losses.append(loss.item())
  with torch.no_grad():
    vloss = mse(predict(x_val, w, b), y_val)
    val_losses.append(vloss.item())

# 7. Evaluate final model
with torch.no_grad():
  final_train = mse(predict(x_train, w, b), y_train).item()
  final_val = mse(predict(x_val, w, b), y_val).item()

print('Known params: w={}, b={}'.format(true_w, true_b))
print('Learned params: w={}, b={}'.format(w.item(), b.item()))
print('Final train MSE:', final_train)
print('Final val MSE:', final_val)

plt.figure(figsize=(5,4))
plt.scatter(x_train.numpy(), y_train.numpy(), s=12, label='train')
plt.scatter(x_val.numpy(), y_val.numpy(), s=20, marker='x', label='val')
xs = torch.linspace(x.min(), x.max(), 100).unsqueeze(1)
with torch.no_grad():
  ys = predict(xs, w, b)
plt.plot(xs.numpy(), ys.numpy(), color='red', label='fit')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/section15a_fresh_linear_regression.png', dpi=120)
plt.show()


Known params: w=2.0, b=-0.5
Learned params: w=1.9865055084228516, b=-0.5324891805648804
Final train MSE: 0.16121122241020203
Final val MSE: 0.14257141947746277


/tmp/ipykernel_69259/1051923849.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
